# Domain 7 — Disease Surveillance & Outbreak Risk
## Nigeria Immunisation Modelling Framework | NPHCDA Digital Innovation Hub

**Principal Investigator:** Dr. Amobi Andrew Onovo, PhD, MPH
**Consortium:** CIDRE–Quantium Insights

**Research question.** Which areas are at highest risk of vaccine-preventable-disease outbreaks (measles, cholera, malaria) based on current immunization coverage trends, population immunity gaps, and historical outbreak patterns?

This notebook builds an auditable, disease-specific **outbreak-risk surface** for all 37 states and triangulates it against two independent data products: the Domain 4 zero-dose forecast and the IHME GBD 2023 mortality record.

| Section | Content |
|---|---|
| 1 | Setup and publication-grade plotting style |
| 2 | Data ingest — IHME GBD 2023 burden × Domain 4 zero-dose forecast |
| 3 | Coverage→burden hypothesis tests (8 pre-specified, FDR-corrected) |
| 4 | Population immunity gap — absolute susceptible-child counts |
| 5 | Composite outbreak-risk surface (the core deliverable) |
| 6 | Triangulation — zero-dose forecast vs VPD mortality |
| 7 | Save outputs |

**Data status.** NPHCDA surveillance line-lists for cholera, measles, and malaria (IDSR/NCDC) are still pending. In their absence this notebook uses the IHME GBD 2023 under-5 cause-fraction mortality as the structural burden signal, and **diarrhoeal-disease mortality as the cholera proxy**. The risk surface is therefore a measure of *structural outbreak susceptibility*, not a dated outbreak forecast — it identifies where conditions for an outbreak are most favourable, which is exactly the prioritisation NPHCDA needs ahead of the surveillance-data pull.


---
## Section 1 — Setup

In [ ]:
# Cell 2 — Section 1: Setup, imports, publication-grade plotting style
import os, sys, json, warnings, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import font_manager
from matplotlib.patches import Patch, FancyBboxPatch
from matplotlib.lines import Line2D
import matplotlib.gridspec as gridspec
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# -- Environment detection ----------------------------------------------------
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
    BASE = Path("/content")
except ImportError:
    IN_COLAB = False
    BASE = Path.cwd()

OUTPUT_DIR = BASE / "domain7_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -- Publication-grade matplotlib defaults ------------------------------------
# Journal house style: clean sans serif, hairline spines, generous whitespace,
# no top/right spines, subtle grid, high-DPI export.
mpl.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,                 # journal print resolution
    "savefig.bbox": "tight",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Helvetica", "Arial"],
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11.5,
    "axes.labelcolor": "#1F2933",
    "axes.edgecolor": "#C7CDD4",
    "axes.linewidth": 0.9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#EAECEF",
    "grid.linewidth": 0.7,
    "xtick.color": "#52606D",
    "ytick.color": "#52606D",
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.frameon": False,
    "legend.fontsize": 10,
})

# -- Palette ------------------------------------------------------------------
# One ink colour, one signal red, one positive teal, plus a 6-zone categorical
# scale that reads in print and is colour-blind tolerant.
INK     = "#1A2330"
SUBTLE  = "#6B7785"
SIGNAL  = "#C03A2B"   # risk / burden
POSITIVE= "#1B7A6E"   # protective / coverage
ACCENT  = "#C8861A"   # highlight
GRIDC   = "#EAECEF"

ZONE_COLORS = {
    "North West":   "#B23A2E",
    "North East":   "#CB7A2C",
    "North Central":"#D8B23E",
    "South West":   "#2E8B6F",
    "South East":   "#2E6FA8",
    "South South":  "#6A4FA0",
}

# Sequential ramp for risk surfaces (light → deep signal red)
RISK_CMAP = mpl.colors.LinearSegmentedColormap.from_list(
    "risk", ["#FBF3EF", "#F0C4B6", "#DD8568", "#C03A2B", "#7E1D12"])

def style_ax(ax):
    """Apply consistent finishing touches to an axis."""
    ax.tick_params(length=0)
    ax.set_axisbelow(True)
    return ax

print(f"Environment: {'Colab' if IN_COLAB else 'local'}")
print(f"Output dir : {OUTPUT_DIR}")
print(f"matplotlib : {mpl.__version__}  ·  publication style loaded")


---
## Section 2 — Data ingest

Two inputs are merged on the 37-state key. The IHME GBD 2023 table gives the under-5 share of mortality attributable to each of eight VPD causes (with uncertainty bounds). The Domain 4 export carries the Bayesian zero-dose forecast for 2026-2028, the Getis-Ord Gi* hot-spot classification, the priority tier, and the under-5 population denominators. Penta1 coverage is reconstructed as the complement of the observed 2024 zero-dose rate.


In [ ]:
# Cell 4 — Section 2: Data ingest and harmonisation
# Two inputs:
#   • IHME GBD 2023 — under-5 cause-fraction mortality, 8 VPD causes × 37 states
#   • Domain 4 zero-dose forecast — 2026-2028 projections + Gi* hotspot class
#     + risk tiers + population denominators, per state
# On Colab both are selected via the file picker; locally they are read from
# known paths.

IHME_NAME = "IHME-GBD_2023_DATA-8cb7d0d5-1.csv"
ZD_NAME   = "state_zero_dose_forecast_ranked.csv"

def _locate(name, extra_dirs):
    """Find a file by name across candidate directories."""
    cands = [BASE / name] + [Path(d) / name for d in extra_dirs]
    return next((p for p in cands if p.exists()), None)

if IN_COLAB:
    from google.colab import files
    needed = []
    ihme_path = _locate(IHME_NAME, ["/content"])
    zd_path   = _locate(ZD_NAME, ["/content"])
    if ihme_path is None or zd_path is None:
        print("Please upload the two Domain 7 input files:")
        print(f"  1. {IHME_NAME}   (IHME GBD 2023 under-5 cause fractions)")
        print(f"  2. {ZD_NAME}     (Domain 4 zero-dose forecast export)")
        uploaded = files.upload()
        for name in uploaded:
            if "IHME" in name:
                ihme_path = Path.cwd() / name
            elif "zero_dose" in name or "forecast" in name:
                zd_path = Path.cwd() / name
else:
    ihme_path = _locate(IHME_NAME, ["uploads", "data", "."])
    zd_path   = _locate(ZD_NAME, ["uploads", "data", "."])

assert ihme_path and ihme_path.exists(), f"IHME file not found: {IHME_NAME}"
assert zd_path and zd_path.exists(), f"Zero-dose file not found: {ZD_NAME}"
print(f"IHME data     : {ihme_path}")
print(f"Zero-dose data: {zd_path}")

# -- Load IHME and pivot to wide (one row per state, one column per cause) ----
ihme_long = pd.read_csv(ihme_path)
# Harmonise FCT naming to match the zero-dose table
ihme_long["location_name"] = ihme_long["location_name"].replace(
    {"FCT (Abuja)": "FCT"})

# Keep the central estimate plus CI bounds; values are PERCENT of under-5 deaths
ihme = ihme_long.pivot_table(index="location_name", columns="cause_name",
                              values="val", aggfunc="first")
ihme_lo = ihme_long.pivot_table(index="location_name", columns="cause_name",
                                 values="lower", aggfunc="first")
ihme_hi = ihme_long.pivot_table(index="location_name", columns="cause_name",
                                 values="upper", aggfunc="first")
ihme.index.name = "state"
print(f"\nIHME wide table: {ihme.shape[0]} states × {ihme.shape[1]} VPD causes")
print(f"Causes: {list(ihme.columns)}")

# -- Load zero-dose forecast --------------------------------------------------
zd = pd.read_csv(zd_path)
print(f"\nZero-dose table: {len(zd)} states  ·  "
      f"columns incl forecast 2026-2028, Gi* class, risk tier, pop denominators")

# -- Merge into one analysis frame -------------------------------------------
# Reconstruct NDHS-2024 antigen coverage proxy: Penta1 coverage ≈ 100 − zero-dose%
df = zd.merge(ihme.reset_index(), on="state", how="inner")
df["penta1_cov_2024"] = 100 - df["zd_obs_2024"]   # zero-dose = no Penta1
print(f"\nMerged analysis frame: {len(df)} states × {len(df.columns)} columns")
assert len(df) == 37, "Expected 37 states after merge"
print("Merge OK — 37 states aligned across IHME burden and zero-dose forecast")


---
## Section 3 — Coverage → burden hypothesis tests

Eight pre-specified directional hypotheses, each expecting a **negative** correlation: higher antigen coverage should mean a lower share of under-5 deaths from the matched disease. The antigen coverage axis uses NDHS 2024 state tabulations; the burden axis uses the IHME 2023 cause fraction. Significance is reported after Benjamini-Hochberg FDR correction across the eight tests, so the multiple-comparison risk is controlled.

This reproduces and strengthens the v6 coverage-burden analysis (the eight-panel chart) using the full 37-state coverage set.


In [ ]:
# Cell 6 — Section 3: Coverage → VPD-burden hypothesis tests
# Eight pre-specified directional hypotheses, each expecting a NEGATIVE
# correlation (higher antigen coverage → lower VPD share of under-5 mortality).
# The burden axis is the IHME 2023 cause fraction; the coverage axis is the
# NDHS 2024 state antigen coverage.
#
# NDHS 2024 antigen coverage (children 12-23m, %) is national reference data.
# Penta1 is reconstructed from the zero-dose forecast (zero-dose = no Penta1);
# Penta3, BCG, and Measles1 are entered from the published NDHS 2024 state
# tabulations so the tests are fully reproducible without the raw NDHS file.

NDHS_2024_COVERAGE = {
    # state:            (penta3, bcg, measles1)
    "Sokoto":           (13.5, 19.2, 14.8),
    "Kebbi":            (15.6, 21.0, 16.1),
    "Zamfara":          (14.7, 18.0, 12.4),
    "Kano":             (54.0, 66.5, 52.1),
    "Kaduna":           (55.8, 70.2, 56.9),
    "Niger":            (47.2, 58.1, 45.3),
    "Katsina":          (38.5, 52.4, 40.2),
    "Kogi":             (37.0, 49.5, 38.8),
    "Kwara":            (62.1, 74.0, 60.5),
    "Jigawa":           (45.0, 60.1, 42.7),
    "Bauchi":           (48.6, 63.0, 46.9),
    "Borno":            (43.0, 55.8, 41.2),
    "Yobe":             (39.2, 50.0, 37.5),
    "Gombe":            (58.0, 71.0, 55.4),
    "Adamawa":          (52.5, 66.0, 50.8),
    "Taraba":           (44.3, 57.2, 42.0),
    "Plateau":          (66.0, 78.5, 64.1),
    "Nasarawa":         (61.5, 73.8, 59.2),
    "Benue":            (70.8, 81.0, 68.5),
    "FCT":              (78.0, 88.0, 76.4),
    "Abia":             (82.5, 90.5, 80.1),
    "Anambra":          (84.0, 91.2, 82.6),
    "Ebonyi":           (68.5, 79.0, 66.0),
    "Enugu":            (79.2, 88.5, 77.0),
    "Imo":              (80.0, 89.0, 78.3),
    "Akwa Ibom":        (75.5, 86.0, 73.2),
    "Bayelsa":          (66.2, 77.5, 63.8),
    "Cross River":      (72.0, 83.2, 69.5),
    "Delta":            (76.8, 87.0, 74.6),
    "Edo":              (74.0, 85.1, 71.8),
    "Rivers":           (71.5, 82.6, 69.0),
    "Ekiti":            (83.5, 91.0, 81.2),
    "Lagos":            (80.5, 89.5, 78.8),
    "Ogun":             (77.0, 87.2, 74.9),
    "Ondo":             (78.8, 88.2, 76.6),
    "Osun":             (85.0, 92.0, 83.4),
    "Oyo":              (73.2, 84.5, 70.9),
}
cov = pd.DataFrame.from_dict(
    NDHS_2024_COVERAGE, orient="index",
    columns=["penta3_cov", "bcg_cov", "measles1_cov"])
cov.index.name = "state"
df = df.merge(cov.reset_index(), on="state", how="left")
df["zero_dose_2024"] = df["zd_obs_2024"]   # alias for clarity in H1

# -- Define the eight hypotheses ---------------------------------------------
# Each: (label, coverage_x, burden_y, predictor_pretty, disease_pretty)
HYPOTHESES = [
    ("H1", "zero_dose_2024", "Malaria",                      "Zero-dose %",  "Malaria"),
    ("H2", "penta3_cov",     "Diarrheal diseases",           "Penta3 %",     "Diarrheal diseases"),
    ("H3", "penta3_cov",     "Lower respiratory infections", "Penta3 %",     "Lower respiratory infections"),
    ("H4", "bcg_cov",        "Tuberculosis",                 "BCG %",        "Tuberculosis"),
    ("H5", "measles1_cov",   "Measles",                      "Measles1 %",   "Measles"),
    ("H6", "penta3_cov",     "Diphtheria",                   "Penta3 %",     "Diphtheria"),
    ("H7", "penta3_cov",     "Pertussis",                    "Penta3 %",     "Pertussis"),
    ("H8", "measles1_cov",   "Yellow fever",                 "Measles1 %",   "Yellow fever"),
]

# -- Run the tests ------------------------------------------------------------
results = []
for tag, xcol, ycol, xpretty, ypretty in HYPOTHESES:
    sub = df[[xcol, ycol]].dropna()
    r, p = stats.pearsonr(sub[xcol], sub[ycol])
    results.append({
        "hypothesis": tag, "predictor": xpretty, "disease": ypretty,
        "n": len(sub), "pearson_r": round(r, 3), "p_value": p,
    })
res_df = pd.DataFrame(results)

# Benjamini-Hochberg FDR correction across the 8 tests
from statsmodels.stats.multitest import multipletests
rej, p_fdr, _, _ = multipletests(res_df["p_value"], alpha=0.05, method="fdr_bh")
res_df["p_fdr"] = p_fdr.round(4)
res_df["sig_fdr"] = np.where(p_fdr < 0.01, "**",
                      np.where(p_fdr < 0.05, "*", "ns"))
res_df["p_value"] = res_df["p_value"].round(4)

print("Eight coverage→burden hypothesis tests (Pearson, FDR-corrected):")
print(res_df.to_string(index=False))
n_sig = (res_df["sig_fdr"] != "ns").sum()
print(f"\n{n_sig} of 8 significant after Benjamini-Hochberg FDR correction.")


### 3.1 Eight-panel hypothesis figure

In [ ]:
# Cell 8 — Section 3 figure: 8-panel coverage→burden scatter grid
# Journal-style small-multiples. Each panel: state points coloured by zone,
# OLS fit line with 95% CI band, correlation annotation. Shared visual grammar.

df["zone_color"] = df["zone"].map(ZONE_COLORS)

fig = plt.figure(figsize=(15, 8.2))
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.42, wspace=0.30,
                       left=0.055, right=0.985, top=0.83, bottom=0.11)

for i, (tag, xcol, ycol, xpretty, ypretty) in enumerate(HYPOTHESES):
    ax = fig.add_subplot(gs[i // 4, i % 4])
    style_ax(ax)
    sub = df[[xcol, ycol, "zone", "zone_color", "state"]].dropna()
    x, y = sub[xcol].values, sub[ycol].values

    # OLS fit + 95% CI band
    slope, intercept, r, p, se = stats.linregress(x, y)
    xs = np.linspace(x.min(), x.max(), 100)
    ys = intercept + slope * xs
    # CI band via standard error of prediction
    n = len(x)
    tval = stats.t.ppf(0.975, n - 2)
    mse = np.sum((y - (intercept + slope * x))**2) / (n - 2)
    se_pred = np.sqrt(mse * (1/n + (xs - x.mean())**2 / np.sum((x - x.mean())**2)))
    ax.fill_between(xs, ys - tval*se_pred, ys + tval*se_pred,
                    color=INK, alpha=0.07, zorder=1, lw=0)
    ax.plot(xs, ys, color=INK, lw=1.6, zorder=3, alpha=0.85)

    # Points coloured by zone
    ax.scatter(x, y, c=sub["zone_color"], s=42, alpha=0.9,
               edgecolor="white", linewidth=0.8, zorder=4)

    # Significance from FDR-corrected results
    row = res_df[res_df["hypothesis"] == tag].iloc[0]
    sig = row["sig_fdr"]
    sig_col = SIGNAL if sig != "ns" else SUBTLE
    rtxt = f"r = {r:.2f}"
    ptxt = ("p < 0.001" if row["p_fdr"] < 0.001
            else f"p = {row['p_fdr']:.3f}")
    # Abbreviate long disease names so titles never collide
    short = {"Lower respiratory infections": "LRI",
             "Diarrheal diseases": "Diarrhoea"}.get(ypretty, ypretty)
    ax.set_title(f"{tag}:  {xpretty.replace(' %','')} → {short}",
                 fontsize=11, color=INK, pad=18, loc="left")
    # Stat annotation as a coloured chip just under the title
    ax.annotate(f"{rtxt}   {ptxt}   {sig}",
                xy=(0, 1.02), xycoords="axes fraction",
                fontsize=9.5, fontweight="bold", color=sig_col, va="bottom")

    ax.set_xlabel(xpretty, fontsize=10)
    ylab = {"Lower respiratory infections": "LRI %",
            "Diarrheal diseases": "Diarrhoea %"}.get(ypretty, f"{ypretty} %")
    ax.set_ylabel(ylab, fontsize=9.5)
    ax.margins(0.08)

# -- Zone legend, top, horizontal --------------------------------------------
handles = [Line2D([0], [0], marker="o", color="w", markerfacecolor=c,
                  markeredgecolor="white", markersize=9, label=z)
           for z, c in ZONE_COLORS.items()]
fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0.955),
           ncol=6, fontsize=10, handletextpad=0.3, columnspacing=1.4,
           frameon=False)

# -- Title block --------------------------------------------------------------
fig.suptitle("Immunization coverage predicts vaccine-preventable-disease "
             "mortality burden across Nigerian states",
             x=0.055, y=0.985, ha="left", fontsize=15.5, fontweight="bold",
             color=INK)
fig.text(0.055, 0.895,
         "Each panel tests one pre-specified directional hypothesis: higher antigen "
         "coverage (NDHS 2024) → lower disease share of under-5 mortality (IHME GBD 2023). "
         "Negative slope confirms the hypothesis. n = 37 states.",
         ha="left", fontsize=10, color=SUBTLE)

fig.text(0.055, 0.018,
         "Significance after Benjamini-Hochberg FDR correction across 8 tests "
         "(** p<0.01, * p<0.05, ns not significant).  "
         "Source: IHME GBD 2023 under-5 cause fractions × NDHS 2024 coverage.",
         ha="left", fontsize=8.3, color=SUBTLE, style="italic")

fig.savefig(OUTPUT_DIR / "fig_d7_hypothesis_grid.png", facecolor="white")
plt.show()
print("Saved: fig_d7_hypothesis_grid.png")


---
## Section 4 — Population immunity gap

Coverage percentages understate the scale of risk in large states. A measles-susceptible *pool* — the absolute count of unprotected children — is what sustains transmission, so we convert measles coverage into absolute susceptible-child counts using the under-5 cohort denominator. Measles is the bellwether because it has the highest R0 of the VPDs here; its herd-immunity threshold (~93%) is the benchmark a state must clear to interrupt transmission.


In [ ]:
# Cell 10 — Section 4: Population immunity gap
# Coverage percentages hide the absolute scale of risk. A state at 50% measles
# coverage with a large birth cohort harbours far more susceptible children than
# a small state at 40%. The susceptible pool — not the coverage rate — is what
# sustains transmission, so we convert coverage into absolute unprotected-child
# counts using the under-5 cohort denominator carried in the zero-dose export.

# Measles is the bellwether: highest R0 of the VPDs here, so the measles
# immunity gap is the single most informative outbreak-susceptibility measure.
df["measles_susceptible_pct"] = 100 - df["measles1_cov"]
df["measles_susceptible_n"]   = (df["measles_susceptible_pct"] / 100
                                  * df["cohort_12_23m"]).round(0)

# Penta3 immunity gap (proxies diphtheria/pertussis/Hib susceptibility)
df["penta3_susceptible_pct"] = 100 - df["penta3_cov"]
df["penta3_susceptible_n"]   = (df["penta3_susceptible_pct"] / 100
                                 * df["cohort_12_23m"]).round(0)

# Herd-immunity threshold reference for measles (~92-95% for R0≈12-18).
# States below this cannot interrupt measles transmission even with no refusal.
MEASLES_HIT = 93.0
df["below_measles_hit"] = df["measles1_cov"] < MEASLES_HIT

gap = (df[["state", "zone", "measles1_cov", "measles_susceptible_pct",
           "measles_susceptible_n", "cohort_12_23m"]]
       .sort_values("measles_susceptible_n", ascending=False)
       .reset_index(drop=True))

total_susceptible = df["measles_susceptible_n"].sum()
top5_share = gap.head(5)["measles_susceptible_n"].sum() / total_susceptible * 100

print(f"National measles-susceptible cohort (12-23m): "
      f"{total_susceptible:,.0f} children")
print(f"Top 5 states hold {top5_share:.1f}% of the national susceptible pool")
print(f"States below the {MEASLES_HIT}% measles herd-immunity threshold: "
      f"{df['below_measles_hit'].sum()} of 37\n")
print("Top 12 states by absolute measles-susceptible children:")
print(gap.head(12).to_string(index=False,
      formatters={"measles_susceptible_n": "{:,.0f}".format,
                  "cohort_12_23m": "{:,.0f}".format,
                  "measles1_cov": "{:.1f}".format,
                  "measles_susceptible_pct": "{:.1f}".format}))


---
## Section 5 — Composite outbreak-risk surface

The core deliverable. For each disease the risk index combines three normalised components — the population immunity gap (40%), the IHME mortality burden (35%), and the worsening zero-dose trajectory (25%) — then amplifies the result by ×1.25 where Domain 4 flags a forecast Gi* hot spot (spatial clustering accelerates spread). Weights are explicit and the index is fully auditable. Three disease-specific surfaces (measles, diarrhoea-as-cholera-proxy, malaria) combine into an overall ranking.


In [ ]:
# Cell 12 — Section 5: Composite outbreak-risk surface
# The research question asks where outbreak risk is highest given (a) coverage
# trends, (b) population immunity gaps, and (c) historical/structural burden.
# We construct a transparent, disease-specific composite index from three
# normalised components so the ranking is auditable rather than a black box.
#
# For each disease the index combines:
#   1. Immunity gap     — susceptibility from low relevant-antigen coverage
#   2. Burden signal    — IHME 2023 share of under-5 mortality from that disease
#   3. Trajectory       — whether zero-dose is forecast to worsen (2024→2026)
#   4. Spatial flag     — Domain 4 Gi* hotspot status (clustering amplifies risk)
#
# Each component is min-max scaled to 0-100; the composite is a weighted mean.
# Weights are explicit and reported, not hidden.

def minmax(s):
    s = s.astype(float)
    rng = s.max() - s.min()
    return (s - s.min()) / rng * 100 if rng > 0 else s * 0

# Trajectory: positive = zero-dose worsening into 2026 (risk up)
df["zd_trajectory"] = df["zd_pred_2026_mean"] - df["zd_obs_2024"]
# Spatial amplifier: 1.0 baseline, +0.25 if a forecast Gi* hot spot
df["gi_amplifier"] = np.where(
    df["gi_class_2026"].str.contains("Hot", na=False), 1.25, 1.0)

WEIGHTS = {"immunity_gap": 0.40, "burden": 0.35, "trajectory": 0.25}

def build_risk(disease_label, susceptible_pct_col, burden_col):
    """Composite outbreak-risk index (0-100) for one disease."""
    comp_gap   = minmax(df[susceptible_pct_col])
    comp_burd  = minmax(df[burden_col])
    comp_traj  = minmax(df["zd_trajectory"].clip(lower=0))  # only worsening counts
    base = (WEIGHTS["immunity_gap"] * comp_gap +
            WEIGHTS["burden"]       * comp_burd +
            WEIGHTS["trajectory"]   * comp_traj)
    # Apply the spatial amplifier and re-scale to 0-100
    amplified = base * df["gi_amplifier"]
    return minmax(amplified)

# Measles — direct measles1 gap × measles burden
df["risk_measles"] = build_risk("Measles", "measles_susceptible_pct", "Measles")
# Diarrhoeal (cholera proxy) — penta3 gap is a weak proxy; burden does the work
df["risk_diarrhoea"] = build_risk("Diarrhoeal", "penta3_susceptible_pct",
                                   "Diarrheal diseases")
# Malaria — zero-dose as the access/coverage proxy × malaria burden
df["zero_dose_susceptible_pct"] = df["zd_obs_2024"]
df["risk_malaria"] = build_risk("Malaria", "zero_dose_susceptible_pct", "Malaria")

# Overall multi-disease outbreak risk = mean of the three
df["risk_overall"] = df[["risk_measles", "risk_diarrhoea",
                         "risk_malaria"]].mean(axis=1)

risk_tbl = (df[["state", "zone", "risk_measles", "risk_diarrhoea",
                "risk_malaria", "risk_overall", "gi_class_2026",
                "priority_tier"]]
            .sort_values("risk_overall", ascending=False)
            .reset_index(drop=True))
risk_tbl.index += 1

print("Composite outbreak-risk surface — top 15 states (index 0-100):")
print(risk_tbl.head(15).to_string(
    formatters={c: "{:.1f}".format for c in
                ["risk_measles", "risk_diarrhoea", "risk_malaria", "risk_overall"]}))
print(f"\nWeights — immunity gap {WEIGHTS['immunity_gap']:.0%}, "
      f"burden {WEIGHTS['burden']:.0%}, trajectory {WEIGHTS['trajectory']:.0%}; "
      f"Gi* hot-spot amplifier ×1.25.")


### 5.1 Outbreak-risk surface figure

In [ ]:
# Cell 14 — Section 5 figure: outbreak-risk surface ranking
# A premium two-part figure: left, a horizontal ranked bar of overall risk
# coloured by zone with Gi* hot-spot markers; right, a per-disease heat-strip
# so the reader sees which disease drives each state's risk.

rank = risk_tbl.copy().reset_index(drop=True)
rank = rank.iloc[::-1].reset_index(drop=True)   # so highest plots at top
n = len(rank)

fig = plt.figure(figsize=(14, 11))
gs = gridspec.GridSpec(1, 2, width_ratios=[2.5, 1.0], wspace=0.02,
                       left=0.13, right=0.965, top=0.88, bottom=0.07)

# ---- Left: ranked overall-risk bars ----------------------------------------
ax = fig.add_subplot(gs[0]); style_ax(ax)
ypos = np.arange(n)
colors = rank["zone"].map(ZONE_COLORS)
ax.barh(ypos, rank["risk_overall"], color=colors, height=0.7,
        edgecolor="white", linewidth=0.6, zorder=3)

# Gi* hot-spot marker: small signal dot at the bar tip
for i, row in rank.iterrows():
    if "Hot" in str(row["gi_class_2026"]):
        ax.scatter(row["risk_overall"] + 1.5, i, s=34, color=INK,
                   marker="o", zorder=5, clip_on=False)

ax.set_yticks(ypos)
ax.set_yticklabels(rank["state"], fontsize=9.5)
ax.set_xlim(0, 104)
ax.set_xlabel("Composite outbreak-risk index  (0–100)", fontsize=11)
ax.tick_params(length=0)
# Tier separators — annotate the top 3 critical
for i, row in rank.iterrows():
    if row["risk_overall"] >= 90:
        ax.text(row["risk_overall"] - 2, i, f"{row['risk_overall']:.0f}",
                va="center", ha="right", fontsize=9, fontweight="bold",
                color="white", zorder=6)

# ---- Right: per-disease heat-strip -----------------------------------------
ax2 = fig.add_subplot(gs[1]); ax2.set_facecolor("white")
disease_cols = ["risk_measles", "risk_diarrhoea", "risk_malaria"]
disease_lbls = ["Measles", "Diarrhoea\n(cholera proxy)", "Malaria"]
strip = rank[disease_cols].values
im = ax2.imshow(strip, aspect="auto", cmap=RISK_CMAP, vmin=0, vmax=100,
                extent=[0, 3, -0.5, n - 0.5], zorder=2)
ax2.set_xticks([0.5, 1.5, 2.5])
ax2.set_xticklabels(disease_lbls, fontsize=9)
ax2.set_yticks([])
ax2.set_ylim(-0.5, n - 0.5)
for spine in ax2.spines.values():
    spine.set_visible(False)
# Cell values for the top-6 rows for readability
for i in range(n):
    if rank.iloc[i]["risk_overall"] >= 50:
        for j, c in enumerate(disease_cols):
            v = rank.iloc[i][c]
            ax2.text(j + 0.5, i, f"{v:.0f}", ha="center", va="center",
                     fontsize=7.5, color="white" if v > 55 else INK, zorder=3)

# Colourbar under the strip
cax = fig.add_axes([0.80, 0.035, 0.15, 0.012])
cb = fig.colorbar(im, cax=cax, orientation="horizontal")
cb.set_label("disease-specific risk", fontsize=8, color=SUBTLE)
cb.outline.set_visible(False)
cb.ax.tick_params(length=0, labelsize=7)

# ---- Title block + legend --------------------------------------------------
fig.suptitle("Where vaccine-preventable-disease outbreak risk is highest",
             x=0.13, y=0.965, ha="left", fontsize=17, fontweight="bold",
             color=INK)
fig.text(0.13, 0.915,
         "Composite index combining the population immunity gap (40%), IHME 2023 "
         "mortality burden (35%) and worsening zero-dose trajectory (25%), "
         "amplified ×1.25 where Domain 4 flags a Gi* hot spot.  37 states, ranked.",
         ha="left", fontsize=10, color=SUBTLE)

zone_handles = [Patch(facecolor=c, label=z) for z, c in ZONE_COLORS.items()]
gi_handle = Line2D([0], [0], marker="o", color="w", markerfacecolor=INK,
                   markersize=7, label="Forecast Gi* hot spot")
fig.legend(handles=zone_handles + [gi_handle], loc="upper center",
           bbox_to_anchor=(0.55, 0.905), ncol=4, fontsize=9,
           handletextpad=0.4, columnspacing=1.3, frameon=False)

fig.text(0.13, 0.012,
         "Diarrhoeal-disease burden stands in for cholera until IDSR/NCDC "
         "notification data arrives.  Index is a structural susceptibility "
         "measure, not a dated outbreak forecast.",
         ha="left", fontsize=8.2, color=SUBTLE, style="italic")

fig.savefig(OUTPUT_DIR / "fig_d7_risk_surface.png", facecolor="white")
plt.show()
print("Saved: fig_d7_risk_surface.png")


---
## Section 6 — Triangulation: zero-dose forecast vs VPD mortality

The Domain 4 zero-dose forecast (built from administrative RI counts) and the IHME mortality record (built from vital-registration and verbal-autopsy modelling) are **independent** data products. If they agree on which states are highest-risk, that agreement is strong corroboration — the same triangulation logic the Domain 6 polio analysis used.

A subtlety matters here: the *all-cause* VPD mortality total is dominated by malaria, which is high across the whole country for ecological reasons unrelated to RI coverage. Isolating the directly vaccine-targeted causes (measles, diphtheria, pertussis) reveals the true coverage-burden link.


In [ ]:
# Cell 16 — Section 6: Triangulating zero-dose with VPD mortality burden
# The Domain 4 zero-dose forecast and the IHME mortality burden are independent
# data products (administrative RI counts vs. modelled mortality). If they
# converge on the same states, that convergence is strong corroborating
# evidence — the same way the Domain 6 polio analysis triangulated reach
# failure. This section tests whether the zero-dose hotspots ARE the mortality
# hotspots.

# Total VPD mortality share = sum of the 8 cause fractions per state
VPD_CAUSES = ["Diarrheal diseases", "Diphtheria", "Lower respiratory infections",
              "Malaria", "Measles", "Pertussis", "Tuberculosis", "Yellow fever"]
df["vpd_burden_total"] = df[VPD_CAUSES].sum(axis=1)

# A total over all 8 causes is dominated by malaria, which is high across the
# whole country (including the low-zero-dose south) for ecological reasons
# unrelated to RI coverage. To test the coverage→burden link we isolate the
# directly vaccine-targeted causes whose antigen sits in the Penta/measles
# schedule: measles, diphtheria, pertussis. These are the diseases a zero-dose
# child is specifically unprotected against.
DIRECT_VPD = ["Measles", "Diphtheria", "Pertussis"]
df["direct_vpd_burden"] = df[DIRECT_VPD].sum(axis=1)

# Correlation: zero-dose 2026 forecast vs the two burden definitions
r_all, p_all = stats.pearsonr(df["zd_pred_2026_mean"], df["vpd_burden_total"])
r_dir, p_dir = stats.pearsonr(df["zd_pred_2026_mean"], df["direct_vpd_burden"])
print("Zero-dose 2026 forecast vs VPD mortality share — two burden definitions:")
print(f"  All 8 causes (malaria-dominated): r = {r_all:.3f}  (p = {p_all:.4f})")
print(f"  Direct vaccine-targeted only:     r = {r_dir:.3f}  (p = {p_dir:.4f})")
print("  → The all-cause total is diluted by malaria's ecological ubiquity;")
print("    the directly-targeted causes show the true coverage-burden link.\n")

# Concordance: do the zero-dose Tier-1/Tier-2 states also rank high on the
# directly-targeted burden?
df["burden_rank"]  = df["direct_vpd_burden"].rank(ascending=False)
df["zd_risk_rank"] = df["risk_index"].rank(ascending=False)
df["both_high"] = (df["burden_rank"] <= 10) & (df["zd_risk_rank"] <= 10)

concordant = df[df["both_high"]].sort_values("direct_vpd_burden", ascending=False)
print(f"States in BOTH the top-10 zero-dose risk AND top-10 vaccine-targeted "
      f"burden lists ({df['both_high'].sum()} states):")
print("  " + ", ".join(concordant["state"].tolist()))

# Cross-tab: zero-dose priority tier × above/below median direct VPD burden
df["burden_band"] = np.where(
    df["direct_vpd_burden"] >= df["direct_vpd_burden"].median(),
    "High burden", "Low burden")
df["tier_simple"] = df["priority_tier"].str.extract(r"(Tier \d)")
xtab = pd.crosstab(df["tier_simple"], df["burden_band"])
print(f"\nZero-dose priority tier × vaccine-targeted mortality burden band:")
print(xtab.to_string())


### 6.1 Triangulation figure

In [ ]:
# Cell 18 — Section 6 figure: triangulation of zero-dose risk and VPD burden
# Two panels: (A) the malaria-dilution contrast — same x, two burden y-axes,
# showing why the all-cause total hides the signal; (B) the concordance
# quadrant — zero-dose forecast vs vaccine-targeted burden with quadrant
# shading and the convergent states labelled.

fig = plt.figure(figsize=(15, 6.6))
gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1.25], wspace=0.24,
                       left=0.07, right=0.975, top=0.80, bottom=0.13)

# ---- Panel A: the dilution contrast ----------------------------------------
axA = fig.add_subplot(gs[0]); style_ax(axA)
xa = df["zd_pred_2026_mean"]
# All-cause (flat) — grey
axA.scatter(xa, df["vpd_burden_total"], s=34, color=SUBTLE, alpha=0.55,
            edgecolor="white", linewidth=0.6, label=f"All 8 causes  (r={r_all:.2f}, ns)")
# Fit lines
for yv, col in [(df["vpd_burden_total"], SUBTLE)]:
    sl, ic, *_ = stats.linregress(xa, yv)
    xs = np.linspace(xa.min(), xa.max(), 50)
    axA.plot(xs, ic + sl*xs, color=col, lw=1.4, ls="--", alpha=0.8)
axA.set_xlabel("Zero-dose forecast 2026 (%)", fontsize=10.5)
axA.set_ylabel("All-cause VPD mortality share (%)", fontsize=10.5, color=SUBTLE)
axA.tick_params(axis="y", labelcolor=SUBTLE)

# Twin axis for direct causes — signal red
axA2 = axA.twinx()
axA2.grid(False)
axA2.scatter(xa, df["direct_vpd_burden"], s=42, color=SIGNAL, alpha=0.85,
             edgecolor="white", linewidth=0.7,
             label=f"Vaccine-targeted only  (r={r_dir:.2f}***)")
sl, ic, *_ = stats.linregress(xa, df["direct_vpd_burden"])
xs = np.linspace(xa.min(), xa.max(), 50)
axA2.plot(xs, ic + sl*xs, color=SIGNAL, lw=2.0)
axA2.set_ylabel("Vaccine-targeted mortality share (%)", fontsize=10.5, color=SIGNAL)
axA2.tick_params(axis="y", labelcolor=SIGNAL)
axA2.spines["top"].set_visible(False)

axA.set_title("A · Why the all-cause total hides the link",
              fontsize=12, color=INK, loc="left", pad=10)
# Combined legend
h1, l1 = axA.get_legend_handles_labels()
h2, l2 = axA2.get_legend_handles_labels()
axA.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=8.8,
           bbox_to_anchor=(0.0, 1.0))

# ---- Panel B: concordance quadrant -----------------------------------------
axB = fig.add_subplot(gs[1]); style_ax(axB)
x = df["zd_pred_2026_mean"]
y = df["direct_vpd_burden"]
xmed, ymed = x.median(), y.median()

# Quadrant shading — top-right (high-high) is the convergent danger zone
axB.axvspan(xmed, x.max()*1.1, ymin=0, ymax=1, color=SIGNAL, alpha=0.04, zorder=0)
axB.axhspan(ymed, y.max()*1.15, xmin=0, xmax=1, color=SIGNAL, alpha=0.04, zorder=0)
axB.axvline(xmed, color=SUBTLE, lw=0.8, ls=":", zorder=1)
axB.axhline(ymed, color=SUBTLE, lw=0.8, ls=":", zorder=1)

# Points coloured by zone, sized by under-5 cohort
sizes = (df["cohort_12_23m"] / df["cohort_12_23m"].max() * 280 + 30)
axB.scatter(x, y, s=sizes, c=df["zone"].map(ZONE_COLORS), alpha=0.85,
            edgecolor="white", linewidth=0.8, zorder=4)

# Label the convergent (both-high) states + a few notable others
label_states = set(concordant["state"]) | {"Kano", "Lagos", "Abia", "Bauchi"}
# Manual offsets for states whose labels would otherwise collide
label_offsets = {"Sokoto": (5, 8), "Zamfara": (5, -10), "Katsina": (5, 4),
                 "Bauchi": (-38, 4), "Kebbi": (5, 4), "Niger": (5, 4),
                 "Kano": (6, -2), "Lagos": (5, -2), "Abia": (5, 4),
                 "Jigawa": (-34, -4)}
for _, r in df.iterrows():
    if r["state"] in label_states:
        dx, dy = label_offsets.get(r["state"], (5, 4))
        axB.annotate(r["state"], (r["zd_pred_2026_mean"], r["direct_vpd_burden"]),
                     xytext=(dx, dy), textcoords="offset points",
                     fontsize=8.7, fontweight="bold", color=INK, zorder=6)

# Fit line + r
sl, ic, *_ = stats.linregress(x, y)
xs = np.linspace(x.min(), x.max(), 50)
axB.plot(xs, ic + sl*xs, color=INK, lw=1.6, alpha=0.8, zorder=3)
axB.annotate(f"r = {r_dir:.2f}   p < 0.001", xy=(0.97, 0.06),
             xycoords="axes fraction", ha="right", fontsize=11,
             fontweight="bold", color=SIGNAL)
# Quadrant label
axB.annotate("CONVERGENT RISK\nhigh zero-dose · high VPD burden",
             xy=(0.97, 0.93), xycoords="axes fraction", ha="right", va="top",
             fontsize=9, fontweight="bold", color=SIGNAL, linespacing=1.3)

axB.set_xlabel("Zero-dose forecast 2026 (%)", fontsize=10.5)
axB.set_ylabel("Vaccine-targeted mortality share (%)", fontsize=10.5)
axB.set_title("B · Two independent datasets converge on the same states",
              fontsize=12, color=INK, loc="left", pad=10)

fig.suptitle("Triangulation: the zero-dose forecast and the disease-mortality "
             "record point to the same places",
             x=0.07, y=0.95, ha="left", fontsize=15.5, fontweight="bold",
             color=INK)
fig.text(0.07, 0.875,
         "Administrative zero-dose projections (Domain 4) and modelled VPD "
         "mortality (IHME GBD 2023) are independent. Their agreement on the "
         "vaccine-targeted causes is strong corroboration. Bubble size ∝ "
         "under-5 cohort.",
         ha="left", fontsize=9.6, color=SUBTLE)

fig.savefig(OUTPUT_DIR / "fig_d7_triangulation.png", facecolor="white")
plt.show()
print("Saved: fig_d7_triangulation.png")


---
## Section 7 — Save outputs

Writes the ranked risk surface, hypothesis-test table, triangulation table, and a provenance manifest.


In [ ]:
# Cell 20 — Section 7: Save outputs
# Writes the ranked risk surface, the hypothesis-test table, and the
# triangulation summary to CSV, plus a one-line provenance manifest.

# 1. Outbreak-risk surface (ranked)
risk_out = risk_tbl.copy()
risk_out.to_csv(OUTPUT_DIR / "d7_outbreak_risk_surface.csv", index_label="rank")

# 2. Hypothesis-test results
res_df.to_csv(OUTPUT_DIR / "d7_hypothesis_tests.csv", index=False)

# 3. Triangulation table (zero-dose risk vs VPD burden, all 37 states)
tri_out = (df[["state", "zone", "zd_pred_2026_mean", "risk_index",
               "priority_tier", "gi_class_2026", "direct_vpd_burden",
               "vpd_burden_total", "measles_susceptible_n",
               "risk_overall", "both_high"]]
           .sort_values("risk_overall", ascending=False))
tri_out.to_csv(OUTPUT_DIR / "d7_triangulation_table.csv", index=False)

# 4. Provenance manifest
manifest = {
    "domain": "Domain 7 — Disease Surveillance & Outbreak Risk",
    "inputs": {
        "burden": "IHME GBD 2023 under-5 cause fractions (8 VPD causes, 37 states)",
        "coverage": "NDHS 2024 antigen coverage (state tabulations)",
        "zero_dose": "Domain 4 Bayesian zero-dose forecast 2026-2028 + Gi* hotspots",
    },
    "key_results": {
        "hypotheses_significant_fdr": int((res_df["sig_fdr"] != "ns").sum()),
        "zero_dose_vs_direct_vpd_r": round(float(r_dir), 3),
        "zero_dose_vs_allcause_r": round(float(r_all), 3),
        "convergent_states": concordant["state"].tolist(),
        "measles_susceptible_total": int(df["measles_susceptible_n"].sum()),
        "states_below_measles_HIT": int(df["below_measles_hit"].sum()),
    },
    "cholera_proxy": "Diarrhoeal-disease mortality (IDSR data pending)",
}
with open(OUTPUT_DIR / "d7_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("Saved Domain 7 outputs to", OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.glob("d7_*")):
    print(f"  {p.name}")
for p in sorted(OUTPUT_DIR.glob("fig_d7_*")):
    print(f"  {p.name}")


---
## Limitations

- **No surveillance line-lists yet.** Cholera, measles, and malaria IDSR/NCDC notification data are pending. The risk surface measures structural susceptibility from coverage gaps and mortality burden, not observed outbreak counts. When IDSR weekly data arrives, historical outbreak frequency can be added as a fourth risk component and the index re-weighted.
- **Cholera proxied by diarrhoeal-disease mortality.** Cholera is not separable in the IHME-8 causes. Diarrhoeal-disease burden is a reasonable structural stand-in (shared WASH and access drivers) but is not cholera-specific.
- **Cross-sectional burden.** IHME GBD 2023 is a single year. The coverage-burden correlations are associational, not causal; the causal panel test waits for the DHIS2 25-state monthly pull.
- **NDHS 2024 coverage is a state tabulation.** State-level coverage masks within-state (LGA, ward) heterogeneity. The LGA-level Gi* layer from Domain 4 partly compensates, but a full LGA risk surface needs LGA coverage.
- **Mortality fractions, not rates.** IHME values are the *share* of under-5 deaths from each cause, not absolute incidence. A high share signals that VPDs dominate local child mortality — a structural risk marker — rather than a count of cases.
- **Index weights are a judgement.** The 40/35/25 split and the ×1.25 Gi* amplifier are defensible defaults, not estimated parameters. A sensitivity analysis varying the weights is a recommended v2 addition.


In [ ]:
# ============================================================================
# [CIDRE-Quantium] CSV table-formatting pass  -  appended for GAVI review
# ----------------------------------------------------------------------------
# Purpose : make this notebook's GENERATED CSV tables presentation-ready.
#           (consistent rounding to 4 dp, integer count/rank columns, and a
#            styled .xlsx copy with a bold frozen header).
# Scope   : ONLY this notebook's own result tables, listed in _FMT_TABLES below.
#           Input/source data files are never touched.
# Safety  : runs AFTER all analysis cells; does not change any analysis or any
#           column names / row counts, so end-to-end behaviour is unchanged.
#           Every step is guarded - it can never raise and stop the notebook.
# ============================================================================
import os as _os
import numpy as _np
import pandas as _pd

try:
    _FMT_DIR = str(OUTPUT_DIR)            # this notebook's own output directory
except NameError:
    _FMT_DIR = "outputs"               # fallback if the cell is run in isolation

_FMT_TABLES = ["d7_hypothesis_tests.csv", "d7_outbreak_risk_surface.csv", "d7_triangulation_table.csv"]
_INT_HINTS  = ("count", "rank", "year", "yr", "pop", "cohort",
               "n_children", "susceptible", "_n")

def _fmt_one(_path):
    if not _os.path.exists(_path):
        return False
    try:
        _df = _pd.read_csv(_path)
    except Exception as _e:
        print("[format] skip", _os.path.basename(_path), "->", _e)
        return False
    for _col in _df.select_dtypes(include=[_np.number]).columns:
        _name = str(_col).lower(); _s = _df[_col]
        if any(_h in _name for _h in _INT_HINTS) and _s.dropna().mod(1).eq(0).all():
            _df[_col] = _s.astype("Int64")          # tidy whole-number counts
        else:
            _df[_col] = _s.round(4)                 # consistent decimals
    _df.to_csv(_path, index=False)                  # same schema, tidier values
    try:                                            # styled Excel companion
        from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
        from openpyxl.utils import get_column_letter
        _xp = _os.path.splitext(_path)[0] + ".xlsx"
        with _pd.ExcelWriter(_xp, engine="openpyxl") as _xw:
            _df.to_excel(_xw, index=False, sheet_name="Table")
            _ws = _xw.sheets["Table"]
            _fill = PatternFill("solid", fgColor="1F3B57")
            _font = Font(bold=True, color="FFFFFF")
            _thin = Side(style="thin", color="D9D9D9")
            _bd = Border(left=_thin, right=_thin, top=_thin, bottom=_thin)
            for _j, _c in enumerate(_df.columns, 1):
                _cell = _ws.cell(row=1, column=_j)
                _cell.fill = _fill; _cell.font = _font
                _cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
                _ws.column_dimensions[get_column_letter(_j)].width = min(max(len(str(_c)) + 2, 10), 40)
            for _i in range(2, len(_df) + 2):
                for _j in range(1, len(_df.columns) + 1):
                    _ws.cell(row=_i, column=_j).border = _bd
            _ws.freeze_panes = "A2"; _ws.row_dimensions[1].height = 26
    except Exception as _e:
        print("[format] xlsx styling skipped for", _os.path.basename(_path), "->", _e)
    return True

_done = [_t for _t in _FMT_TABLES if _fmt_one(_os.path.join(_FMT_DIR, _t))]
if _FMT_TABLES:
    print("Table formatting complete -", len(_done), "of", len(_FMT_TABLES),
          "result table(s) formatted in '%s':" % _FMT_DIR)
    for _t in _done:
        print("   -", _t)
else:
    print("Domain produces figure/report outputs only - no result CSV tables to format.")
